# Notebook 16 – Feature Engineering Mini Challenge

## Challenge Objective

The objective of this mini challenge is to independently understand a dataset and improve it through feature engineering.

Tasks:
1. Understand the dataset.
2. Identify the target variable.
3. Identify existing features.
4. Identify potential feature gaps.
5. Create at least 10 meaningful features.
6. Explain every created feature.
7. Check for feature leakage.
8. Remove irrelevant features.
9. Perform feature selection.
10. Compare the dataset before and after feature engineering.

The final goal is to prepare a useful dataset for machine learning.



## Step 1 – Understand

Dataset understanding is the first step in a machine learning project.

Before creating new features, we need to understand:
- Dataset size.
- Column names.
- Data types.
- Missing values.
- Duplicate records.
- Numerical columns.
- Categorical columns.

Understanding the dataset helps us decide which features can be created.



## Step 2 – Demonstrate

For a customer churn dataset, possible information includes:

- Customer tenure.
- Monthly charges.
- Total charges.
- Contract type.
- Internet service.
- Customer family information.

By examining these columns, we can identify useful relationships.

For example, tenure and monthly charges can be combined to create a new numerical feature.



## Step 3 – Implement

Load the cleaned Sprint 5 dataset.

Inspect its shape, columns, data types, missing values, duplicate records, and sample rows.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from tkinter import Tk, filedialog

# Select the cleaned Sprint 5 dataset
root = Tk()
root.withdraw()

file_path = filedialog.askopenfilename(
    title="Select Cleaned Sprint 5 Dataset",
    filetypes=[("CSV Files", "*.csv")]
)

root.destroy()

if not file_path:
    raise FileNotFoundError("Please select a CSV dataset.")

df = pd.read_csv(file_path)

print("Dataset Loaded Successfully!")
print("File Name:", Path(file_path).name)
print("Dataset Shape:", df.shape)

print("\nFirst Five Rows:")
display(df.head())

print("\nDataset Information:")
df.info()

print("\nMissing Values:")
display(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nStatistical Summary:")
display(df.describe(include="all").T)

### Output Explanation

The output displays the basic structure and quality of the dataset.

- Shape shows the number of rows and columns.
- Head displays sample records.
- Info shows column names and data types.
- Missing values identifies incomplete columns.
- Duplicate rows identifies repeated records.
- Statistical summary describes numerical and categorical data.

This information helps us understand the dataset before feature engineering.

# Part 2 – Identify the Target and Existing Features


## Step 1 – Understand

The target variable is the column that the machine learning model needs to predict.

The remaining useful columns are called input features.

For customer churn prediction:
- Target: Churn
- Input features: tenure, MonthlyCharges, Contract, InternetService, and other customer information.

The target must be kept separate from input features.



## Step 2 – Demonstrate

Example:

A telecom company wants to predict whether a customer will leave.

Target:
- Churn

Existing features:
- tenure
- MonthlyCharges
- TotalCharges
- Contract
- InternetService

The model learns patterns from the input features to predict the target.



## Step 3 – Implement

Identify the target column and separate it from the input features.

For this challenge, we use Churn as the target variable.

If the dataset uses a different target column, update the target name below.

In [ ]:
target_column = "Churn"

if target_column not in df.columns:
    raise KeyError(
        "Churn column not found. "
        "Update target_column with the correct target name."
    )

y = df[target_column].copy()
X_original = df.drop(columns=[target_column]).copy()

print("Target Column:", target_column)
print("\nTarget Distribution:")
display(y.value_counts(dropna=False))

print("\nExisting Input Features:")
print(X_original.columns.tolist())

print("\nNumber of Existing Features:", X_original.shape[1])

### Output Explanation

The output identifies the target variable and existing input features.

- y contains the target values.
- X_original contains the input features.
- Target distribution shows the number of records in each target category.

The target is kept separate to prevent it from becoming an input feature.

# Part 3 – Identify Potential Feature Gaps


## Step 1 – Understand

A feature gap is an opportunity to create additional useful information from existing columns.

We should ask:

- Can existing numerical columns be combined?
- Can existing categories be grouped?
- Can customer behaviour be represented more clearly?
- Are there useful relationships between different columns?

New features should have a clear business meaning and should not use future information.




## Step 2 – Demonstrate

Consider the following existing columns:

- tenure
- MonthlyCharges
- TotalCharges
- Contract
- InternetService
- Partner
- Dependents

Potential feature gaps:

1. Tenure can be converted into years.
2. Tenure can be grouped into customer duration categories.
3. Partner and Dependents can identify family connections.
4. Contract can identify month-to-month customers.
5. InternetService can identify customers with internet service.
6. Service columns can be combined to calculate total services.
7. Contract and InternetService can be combined into one category.

These features provide additional customer information.



## Step 3 – Implement

Check which source columns are available.

Identify the features that can be created from the available columns.

Do not create features if their required source columns are missing.

In [ ]:
feature_requirements = {
    "TenureYears": ["tenure"],
    "TenureGroup": ["tenure"],
    "AverageChargesPerMonth": ["TotalCharges", "tenure"],
    "HasFamily": ["Partner", "Dependents"],
    "IsMonthToMonth": ["Contract"],
    "HasInternet": ["InternetService"],
    "HasOnlineSecurity": ["OnlineSecurity"],
    "TotalServices": [
        "PhoneService",
        "MultipleLines",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies"
    ],
    "MonthlyChargePerService": ["MonthlyCharges"],
    "ContractInternetCombination": ["Contract", "InternetService"],
    "PaperlessBillingFlag": ["PaperlessBilling"]
}

available_columns = X_original.columns.tolist()

for feature, required_columns in feature_requirements.items():
    missing = [
        column for column in required_columns
        if column not in available_columns
    ]

    if not missing:
        print(f"{feature}: Can be created")
    else:
        print(f"{feature}: Missing columns {missing}")

### Output Explanation

The output identifies which potential features can be created.

Features with all required source columns are suitable for implementation.

Features with missing source columns should be skipped or redesigned.

This is an important independent problem-solving step because the feature engineering process depends on the actual dataset.

# Part 4 – Create at Least 10 Meaningful Features



## Step 1 – Understand

Feature engineering creates new information from existing dataset columns.

In this challenge, we will create numerical, categorical, binary, and interaction features.

Each feature must:
- Have a clear meaning.
- Use available source columns.
- Be useful for the business problem.
- Avoid using the target variable.
- Avoid using future information.



## Step 2 – Demonstrate

Example:

A customer has:
- Tenure = 24 months
- MonthlyCharges = 100
- TotalCharges = 2400
- Contract = Month-to-month
- InternetService = Fiber optic

We can create:

- TenureYears = 2
- AverageChargesPerMonth = 100
- IsMonthToMonth = 1
- HasInternet = 1

These features represent customer information in a more useful format.



## Step 3 – Implement

Create at least 10 features using the existing columns.

The code checks whether each required column exists before creating a feature.

No feature will be created using the Churn target column.

In [ ]:
# Create a separate copy for feature engineering
engineered_df = X_original.copy()

created_features = []

# 1. TenureYears
if "tenure" in engineered_df.columns:
    tenure = pd.to_numeric(
        engineered_df["tenure"], errors="coerce"
    )

    engineered_df["TenureYears"] = tenure / 12
    created_features.append("TenureYears")


# 2. TenureGroup
if "tenure" in engineered_df.columns:
    tenure = pd.to_numeric(
        engineered_df["tenure"], errors="coerce"
    )

    engineered_df["TenureGroup"] = pd.cut(
        tenure,
        bins=[-1, 12, 48, float("inf")],
        labels=["New", "Medium-Term", "Long-Term"]
    )

    created_features.append("TenureGroup")


# 3. AverageChargesPerMonth
if {"TotalCharges", "tenure"}.issubset(engineered_df.columns):
    total_charges = pd.to_numeric(
        engineered_df["TotalCharges"], errors="coerce"
    )

    tenure = pd.to_numeric(
        engineered_df["tenure"], errors="coerce"
    )

    engineered_df["AverageChargesPerMonth"] = (
        total_charges / tenure.replace(0, np.nan)
    )

    created_features.append("AverageChargesPerMonth")


# 4. HasFamily
if {"Partner", "Dependents"}.issubset(engineered_df.columns):
    engineered_df["HasFamily"] = (
        (engineered_df["Partner"] == "Yes")
        | (engineered_df["Dependents"] == "Yes")
    ).astype(int)

    created_features.append("HasFamily")


# 5. IsMonthToMonth
if "Contract" in engineered_df.columns:
    engineered_df["IsMonthToMonth"] = (
        engineered_df["Contract"] == "Month-to-month"
    ).astype(int)

    created_features.append("IsMonthToMonth")


# 6. HasInternet
if "InternetService" in engineered_df.columns:
    engineered_df["HasInternet"] = (
        engineered_df["InternetService"].notna()
        & (engineered_df["InternetService"] != "No")
    ).astype(int)

    created_features.append("HasInternet")


# 7. HasOnlineSecurity
if "OnlineSecurity" in engineered_df.columns:
    engineered_df["HasOnlineSecurity"] = (
        engineered_df["OnlineSecurity"] == "Yes"
    ).astype(int)

    created_features.append("HasOnlineSecurity")


# 8. TotalServices
service_columns = [
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

available_service_columns = [
    column for column in service_columns
    if column in engineered_df.columns
]

if available_service_columns:
    service_data = engineered_df[
        available_service_columns
    ].eq("Yes").astype(int)

    # Include internet service as one service
    if "InternetService" in engineered_df.columns:
        internet_service = (
            engineered_df["InternetService"].notna()
            & (engineered_df["InternetService"] != "No")
        ).astype(int)

        service_data["InternetService"] = internet_service

    engineered_df["TotalServices"] = service_data.sum(axis=1)

    created_features.append("TotalServices")


# 9. MonthlyChargePerService
if {
    "MonthlyCharges",
    "TotalServices"
}.issubset(engineered_df.columns):

    monthly_charges = pd.to_numeric(
        engineered_df["MonthlyCharges"], errors="coerce"
    )

    engineered_df["MonthlyChargePerService"] = (
        monthly_charges
        / engineered_df["TotalServices"].replace(0, np.nan)
    )

    created_features.append("MonthlyChargePerService")


# 10. ContractInternetCombination
if {"Contract", "InternetService"}.issubset(engineered_df.columns):
    engineered_df["ContractInternetCombination"] = (
        engineered_df["Contract"].fillna("Unknown").astype(str)
        + "_"
        + engineered_df["InternetService"].fillna("Unknown").astype(str)
    )

    created_features.append("ContractInternetCombination")


# 11. PaperlessBillingFlag
if "PaperlessBilling" in engineered_df.columns:
    engineered_df["PaperlessBillingFlag"] = (
        engineered_df["PaperlessBilling"] == "Yes"
    ).astype(int)

    created_features.append("PaperlessBillingFlag")


print("Created Features:")
for number, feature in enumerate(created_features, start=1):
    print(f"{number}. {feature}")

print("\nTotal Features Created:", len(created_features))

print("\nEngineered Dataset Shape:", engineered_df.shape)

display(engineered_df[created_features].head())

### Output Explanation

The output displays all successfully created features.

The notebook attempts to create 11 features.

The actual number may be lower if some source columns are missing.

Feature meanings:

1. TenureYears: Converts customer tenure from months into years.
2. TenureGroup: Groups customers into New, Medium-Term, and Long-Term.
3. AverageChargesPerMonth: Calculates average monthly charges using total charges and tenure.
4. HasFamily: Identifies customers with a partner or dependents.
5. IsMonthToMonth: Identifies month-to-month contract customers.
6. HasInternet: Identifies customers who have internet service.
7. HasOnlineSecurity: Identifies customers who have online security.
8. TotalServices: Counts the number of subscribed services.
9. MonthlyChargePerService: Calculates average monthly charges per subscribed service.
10. ContractInternetCombination: Combines contract type and internet service.
11. PaperlessBillingFlag: Converts paperless billing into a binary feature.

These features provide additional customer information for churn prediction.

# Part 5 – Explain Every Created Feature



## Step 1 – Understand

Feature documentation explains the purpose of each newly created feature.

It should describe:
- Feature name.
- Source columns.
- Calculation or transformation.
- Business meaning.
- Expected usefulness for machine learning.

Documentation helps other developers understand why a feature was created.




## Step 2 – Demonstrate

Example:

Feature: TenureYears

Source: tenure

Calculation: tenure / 12

Business meaning: Customer duration in years.

Reason: It provides customer tenure in a more understandable unit.

Every created feature should have a similar explanation.




## Step 3 – Implement

Create a feature explanation table for all features successfully created.

Save the explanation table as a CSV file.

In [ ]:
feature_explanations = {
    "TenureYears": {
        "Source": "tenure",
        "Method": "tenure / 12",
        "BusinessMeaning": "Customer duration in years",
        "Reason": "Makes customer tenure easier to interpret"
    },
    "TenureGroup": {
        "Source": "tenure",
        "Method": "Binning",
        "BusinessMeaning": "Customer duration category",
        "Reason": "Groups customers by their relationship duration"
    },
    "AverageChargesPerMonth": {
        "Source": "TotalCharges, tenure",
        "Method": "TotalCharges / tenure",
        "BusinessMeaning": "Average monthly customer charges",
        "Reason": "Represents average customer spending"
    },
    "HasFamily": {
        "Source": "Partner, Dependents",
        "Method": "Logical OR",
        "BusinessMeaning": "Whether the customer has family connections",
        "Reason": "Adds family-related customer information"
    },
    "IsMonthToMonth": {
        "Source": "Contract",
        "Method": "Binary condition",
        "BusinessMeaning": "Whether the customer has a month-to-month contract",
        "Reason": "Represents contract flexibility"
    },
    "HasInternet": {
        "Source": "InternetService",
        "Method": "Binary condition",
        "BusinessMeaning": "Whether the customer has internet service",
        "Reason": "Identifies internet-service customers"
    },
    "HasOnlineSecurity": {
        "Source": "OnlineSecurity",
        "Method": "Binary condition",
        "BusinessMeaning": "Whether the customer has online security",
        "Reason": "Identifies customers using security services"
    },
    "TotalServices": {
        "Source": "Service columns",
        "Method": "Count of subscribed services",
        "BusinessMeaning": "Total number of services used",
        "Reason": "Represents customer service usage"
    },
    "MonthlyChargePerService": {
        "Source": "MonthlyCharges, TotalServices",
        "Method": "MonthlyCharges / TotalServices",
        "BusinessMeaning": "Average monthly charge per service",
        "Reason": "Compares customer spending with service usage"
    },
    "ContractInternetCombination": {
        "Source": "Contract, InternetService",
        "Method": "Category combination",
        "BusinessMeaning": "Combined contract and internet category",
        "Reason": "Captures a relationship between two categories"
    },
    "PaperlessBillingFlag": {
        "Source": "PaperlessBilling",
        "Method": "Binary condition",
        "BusinessMeaning": "Whether paperless billing is enabled",
        "Reason": "Represents customer billing preference"
    }
}

explanation_rows = []

for feature in created_features:
    details = feature_explanations[feature]

    explanation_rows.append({
        "Feature": feature,
        **details
    })

feature_explanation_df = pd.DataFrame(explanation_rows)

display(feature_explanation_df)

feature_explanation_df.to_csv(
    "Mini_Challenge_Feature_Explanations.csv",
    index=False
)

print("Feature explanations saved successfully!")

### Output Explanation

The output contains the explanation of every feature created in this challenge.

The documentation includes:
- Source columns.
- Feature creation method.
- Business meaning.
- Reason for creating the feature.

The file Mini_Challenge_Feature_Explanations.csv stores these explanations for future reference.

# Part 6 – Check for Feature Leakage


## Step 1 – Understand

Feature leakage happens when a model receives information that would not be available at the time of prediction.

Examples:
- Including the target column in input features.
- Using customer cancellation information recorded after cancellation.
- Using future transaction information.
- Using test data to calculate training statistics.

Leakage can produce misleadingly high model performance.



## Step 2 – Demonstrate

Suppose we want to predict customer churn.

The Churn column tells us whether a customer has already left.

If we include Churn as an input feature, the model already knows the answer.

Similarly, CancellationDate may only become available after the customer cancels.

Such columns should not be used as prediction inputs.




## Step 3 – Implement

Perform leakage checks.

Check:
- Whether the target appears in input features.
- Whether suspicious future-information columns exist.
- Whether customer identifiers are included.
- Whether newly created features use the target column.

Review suspicious columns before removing them.



In [ ]:
print("FEATURE LEAKAGE CHECKS")
print("-" * 45)

# Check target separation
print(
    "Target included in engineered features:",
    target_column in engineered_df.columns
)

# Check suspicious future-related column names
suspicious_keywords = [
    "cancellationdate",
    "terminationdate",
    "cancellationreason",
    "terminationreason",
    "churndate",
    "churnstatus",
    "future"
]

suspicious_columns = [
    column for column in engineered_df.columns
    if any(keyword in column.lower() for keyword in suspicious_keywords)
]

print("\nSuspicious Columns:")
print(suspicious_columns)

# Check whether created features contain target information
print(
    "\nTarget used in feature creation:",
    target_column in X_original.columns
)

# Check identifier columns
identifier_columns = [
    column for column in engineered_df.columns
    if column.lower() in ["customerid", "customer_id", "id"]
]

print("\nIdentifier Columns:", identifier_columns)

print("\nLeakage Review:")
if suspicious_columns:
    print("Review suspicious columns before model training.")
else:
    print("No obvious future-related columns found.")

print("Target is maintained separately from engineered features.")

### Output Explanation

The output identifies common feature leakage risks.

- The target must not be part of the input features.
- Suspicious future-related columns require manual review.
- Identifier columns are usually not useful model inputs.
- Newly created features should use only information available at prediction time.

These checks identify common leakage risks but do not guarantee that every feature is leakage-free.

# Part 7 – Remove Irrelevant Features


## Step 1 – Understand

Irrelevant features are columns that do not provide useful information for the prediction task.

Examples:
- Customer ID.
- Constant columns.
- Duplicate columns.
- Future-information columns.
- Unnecessary identifiers.

Removing these columns can reduce noise and simplify the machine learning dataset.



## Step 2 – Demonstrate

Consider a dataset containing:

- customerID
- tenure
- MonthlyCharges
- Contract
- Churn

The customerID column identifies a customer but does not directly describe customer behaviour.

Therefore, customerID can be removed from the model input features.

The Churn target must remain separate.



## Step 3 – Implement

Remove:
- Identifier columns.
- Constant columns.
- Suspicious future-information columns identified during leakage review.

Use the same columns for the full dataset before splitting it into training and testing data.

In [ ]:
# Start with engineered features
model_df = engineered_df.copy()

# Identify constant columns
constant_columns = [
    column
    for column in model_df.columns
    if model_df[column].nunique(dropna=False) <= 1
]

# Combine irrelevant columns
columns_to_remove = list(set(
    identifier_columns
    + constant_columns
    + suspicious_columns
))

# Remove irrelevant columns
model_df = model_df.drop(
    columns=columns_to_remove,
    errors="ignore"
)

print("Removed Columns:")
print(columns_to_remove)

print("\nRemaining Features:")
print(model_df.columns.tolist())

print("\nOriginal Engineered Shape:", engineered_df.shape)
print("After Removing Irrelevant Features:", model_df.shape)

### Output Explanation

The output displays the columns removed from the engineered dataset.

- Identifier columns are removed.
- Constant columns are removed because they contain no variation.
- Suspicious future-information columns are excluded after review.

The remaining columns are candidate input features for machine learning.

# Part 8 – Perform Feature Selection


## Step 1 – Understand

Feature selection identifies the most useful input features for machine learning.

It helps:
- Reduce the number of input features.
- Remove less informative features.
- Reduce unnecessary complexity.
- Improve model efficiency.

In this challenge, we will use mutual information with SelectKBest.

Feature selection must be fitted using training data only.



## Step 2 – Demonstrate

Suppose the dataset contains 25 input features.

Some features may provide useful information about customer churn, while others may provide very little information.

SelectKBest evaluates the relationship between each feature and the target using mutual information.

It then selects the highest-scoring features.

Categorical columns must be encoded before feature selection.



## Step 3 – Implement

Perform the following operations:

1. Split the dataset into training and testing data.
2. Separate numerical and categorical features.
3. Fill missing values.
4. Scale numerical features.
5. Encode categorical features.
6. Fit SelectKBest on training data.
7. Transform testing data using the fitted selector.

This prevents test-data information from influencing feature selection.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif

# Separate input and target
X = model_df.copy()
y = df[target_column].copy()

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

# Identify numerical and categorical features
numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\nNumerical Features:", numeric_features)
print("\nCategorical Features:", categorical_features)

# Numerical preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Fit preprocessing on training data only
X_train_processed = preprocessor.fit_transform(X_train)

# Transform testing data
X_test_processed = preprocessor.transform(X_test)

# Select useful features
total_encoded_features = X_train_processed.shape[1]
k = min(20, total_encoded_features)

feature_selector = SelectKBest(
    score_func=mutual_info_classif,
    k=k
)

X_train_selected = feature_selector.fit_transform(
    X_train_processed,
    y_train
)

X_test_selected = feature_selector.transform(
    X_test_processed
)

# Get selected feature names
encoded_feature_names = preprocessor.get_feature_names_out()

selected_feature_names = encoded_feature_names[
    feature_selector.get_support()
]

print("\nOriginal Encoded Feature Count:", total_encoded_features)
print("Selected Feature Count:", len(selected_feature_names))

print("\nSelected Features:")
print(selected_feature_names.tolist())

### Output Explanation

The output displays the number of features before and after feature selection.

- X_train_selected contains the selected training features.
- X_test_selected contains the corresponding selected testing features.
- selected_feature_names contains the names of the selected features.

Mutual information measures how much information a feature provides about the target.

The selector is fitted only on training data to prevent data leakage.

# Part 9 – Compare Dataset Before and After Feature Engineering


## Step 1 – Understand

Dataset comparison helps us understand the impact of feature engineering.

We compare:

Before feature engineering:
- Original number of rows.
- Original number of columns.
- Original feature count.

After feature engineering:
- Number of rows.
- Number of columns.
- Number of newly created features.
- Number of selected features.

This comparison helps evaluate whether the dataset has become more informative.




## Step 2 – Demonstrate

Example:

Before feature engineering:
- 7,043 rows.
- 20 input features.

After feature engineering:
- 7,043 rows.
- 31 input features.

After feature selection:
- Only the most informative encoded features are retained.

The actual counts depend on the selected dataset and available columns.




## Step 3 – Implement

Compare the original dataset, engineered dataset, cleaned model dataset, and selected feature dataset.

Display:
- Row count.
- Column count.
- Number of new features.
- Number of selected features.

In [ ]:
comparison_data = {
    "Dataset Stage": [
        "Original Dataset",
        "Engineered Dataset",
        "After Removing Irrelevant Features",
        "After Feature Selection"
    ],
    "Rows": [
        len(X_original),
        len(engineered_df),
        len(model_df),
        X_train_selected.shape[0]
    ],
    "Features": [
        X_original.shape[1],
        engineered_df.shape[1],
        model_df.shape[1],
        X_train_selected.shape[1]
    ]
}

comparison_df = pd.DataFrame(comparison_data)

print("Dataset Comparison:")
display(comparison_df)

print("\nNew Features Created:", len(created_features))
print("Irrelevant Features Removed:", len(columns_to_remove))
print("Selected Features:", X_train_selected.shape[1])

### Output Explanation

The output compares the dataset at different stages.

- Original Dataset shows the original input features.
- Engineered Dataset includes newly created features.
- After Removing Irrelevant Features shows the cleaned candidate features.
- After Feature Selection shows the selected encoded features from the training dataset.

The row count of the final selected dataset represents the training split, not the full dataset.

The comparison demonstrates how feature engineering and feature selection change the dataset.

# Part 10 – Visualize the Comparison



## Step 1 – Understand

Data visualization makes dataset comparisons easier to understand.

A bar chart can show how the number of features changes at each stage.

This helps us understand the effect of feature creation and feature selection.



## Step 2 – Demonstrate

For example:

- Feature engineering increases the number of available features.
- Removing irrelevant features reduces unnecessary columns.
- Feature selection retains a smaller set of useful features.

A bar chart helps visualize these changes.



## Step 3 – Implement

Create a bar chart to compare the feature count at each stage of the workflow.

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    comparison_df["Dataset Stage"],
    comparison_df["Features"]
)

plt.xlabel("Dataset Stage")
plt.ylabel("Number of Features")
plt.title("Feature Count Before and After Feature Engineering")

plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

### Output Explanation

The bar chart shows the number of features at each stage.

Feature engineering can increase the number of features.

Removing irrelevant features and performing feature selection can reduce the number of features.

The final result should contain useful features suitable for machine learning.

# Part 11 – Save the Final Challenge Outputs



## Step 1 – Understand

Saving the final outputs makes the feature engineering workflow reusable.

We should save:
- Engineered dataset.
- Final selected training features.
- Final selected testing features.
- Training target.
- Testing target.
- Feature explanations.
- Dataset comparison.


## Step 2 – Demonstrate

The final outputs can be reused in the next machine learning workflow.

The selected training features can be used to train a model.

The selected testing features can be used to evaluate the model.

The target values remain separate from the input features.



## Step 3 – Implement

Convert selected feature arrays into DataFrames.

Save the final datasets and documentation as CSV files.

In [ ]:
# Convert sparse arrays into regular NumPy arrays when necessary
if hasattr(X_train_selected, "toarray"):
    train_array = X_train_selected.toarray()
else:
    train_array = np.asarray(X_train_selected)

if hasattr(X_test_selected, "toarray"):
    test_array = X_test_selected.toarray()
else:
    test_array = np.asarray(X_test_selected)

# Create selected feature DataFrames
X_train_final = pd.DataFrame(
    train_array,
    columns=selected_feature_names,
    index=X_train.index
)

X_test_final = pd.DataFrame(
    test_array,
    columns=selected_feature_names,
    index=X_test.index
)

# Save outputs
engineered_df.to_csv(
    "Challenge_Engineered_Dataset.csv",
    index=False
)

X_train_final.to_csv(
    "Challenge_Final_Training_Features.csv",
    index=False
)

X_test_final.to_csv(
    "Challenge_Final_Testing_Features.csv",
    index=False
)

y_train.to_frame(name=target_column).to_csv(
    "Challenge_Training_Target.csv",
    index=False
)

y_test.to_frame(name=target_column).to_csv(
    "Challenge_Testing_Target.csv",
    index=False
)

feature_explanation_df.to_csv(
    "Mini_Challenge_Feature_Explanations.csv",
    index=False
)

comparison_df.to_csv(
    "Mini_Challenge_Dataset_Comparison.csv",
    index=False
)

print("All Challenge Outputs Saved Successfully!")

print("\nFinal Training Features:", X_train_final.shape)
print("Final Testing Features:", X_test_final.shape)

print("\nSaved Files:")
print("1. Challenge_Engineered_Dataset.csv")
print("2. Challenge_Final_Training_Features.csv")
print("3. Challenge_Final_Testing_Features.csv")
print("4. Challenge_Training_Target.csv")
print("5. Challenge_Testing_Target.csv")
print("6. Mini_Challenge_Feature_Explanations.csv")
print("7. Mini_Challenge_Dataset_Comparison.csv")

### Output Explanation

The output confirms that the challenge datasets and documentation have been saved.

The final training and testing features contain the selected, encoded, and processed input features.

The target values are saved separately.

The saved files can be reused for machine learning model training and evaluation.

# Final Conclusion – Feature Engineering Mini Challenge

In this mini challenge, I independently explored the dataset and performed feature engineering.

Completed activities:

1. Understood the dataset structure.
2. Identified the target variable.
3. Identified existing input features.
4. Identified potential feature gaps.
5. Created meaningful numerical and categorical features.
6. Explained every created feature.
7. Checked for possible feature leakage.
8. Removed irrelevant features.
9. Performed feature selection using mutual information.
10. Compared the dataset before and after feature engineering.
11. Saved the final ML-ready datasets and documentation.

## Final Workflow

Original Dataset
        ↓
Dataset Understanding
        ↓
Feature Gap Identification
        ↓
Feature Engineering
        ↓
Feature Explanation
        ↓
Leakage Checking
        ↓
Irrelevant Feature Removal
        ↓
Feature Selection
        ↓
Final ML-Ready Dataset

## Conclusion

Feature engineering improves the information available to machine learning models.

This challenge demonstrates independent dataset exploration, feature creation, leakage awareness, feature selection, and problem-solving skills.